# Análisis Exploratorio de Datos (EDA) — Datos Satelitales GEE
## Proyecto: Red Neuronal Recurrente para Monitoreo de Vegetación (Río Combeima)

> **Metodología CRISP-DM:** Fase 2 — Comprensión de los Datos (Data Understanding)  
> **Fuente de Datos:** Google Earth Engine (MODIS/061/MOD13A2 — 2015-2024)  
> **Área de Estudio:** Cuenca del río Combeima (Tolima, Colombia)  
> **Objetivo:** Caracterizar las propiedades estadísticas, espectrales, temporales y geoespaciales de las series multitemporales para fundamentar el diseño y entrenamiento de redes neuronales recurrentes (LSTM, GRU, BiGRU, Attention, TCN).


---
## 1. Configuración del Entorno y Carga de Datos (Tarea 5.1)
En esta sección importamos las librerías analíticas esenciales, configuramos la reproducibilidad mediante semillas pseudoaleatorias fijas (`random_state=42`) y cargamos el dataset satelital multitemporal extraído desde Google Earth Engine.


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configuración de reproducibilidad y visualización
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 120,
    'font.sans-serif': 'DejaVu Sans',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 0.8,
    'grid.color': '#eeeeee'
})
sns.set_theme(style='whitegrid', palette='viridis')

# Rutas de datos y reportes
DATA_PATH = Path('../data/raw/series_multisensor_combeima.csv')
REPORTS_DIR = Path('../reports/eda')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Carga de datos
df = pd.read_csv(DATA_PATH, parse_dates=['fecha'])
print(f"Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()


In [ ]:
# Validación de Integridad Estructural y Tipos de Datos
print("=== INFORMACIÓN DE COLUMNAS Y TIPOS ===")
print(df.info())

print("\n=== VALORES FALTANTES (NULOS) ===")
print(df.isnull().sum())

print("\n=== RANGOS ESPACIO-TEMPORALES ===")
print(f"Periodo Temporal: {df['fecha'].min().strftime('%Y-%m-%d')} a {df['fecha'].max().strftime('%Y-%m-%d')}")
print(f"Fechas únicas: {df['fecha'].nunique()}")
print(f"Píxeles geográficos únicos (lon, lat): {df.groupby(['longitude', 'latitude']).ngroups}")
print(f"Longitud: [{df['longitude'].min():.4f}, {df['longitude'].max():.4f}]")
print(f"Latitud: [{df['latitude'].min():.4f}, {df['latitude'].max():.4f}]")


### 1.1 Creación de Clases de Cobertura Vegetal (Derivación Espectral)
Dado que el problema de modelado predictivo central es de **regresión de series temporales** (predecir NDVI/EVI en $t+h$), derivamos adicionalmente una variable categórica `clase_cobertura` siguiendo los umbrales estándar de la literatura biofísica y de teledetección. Esto permite realizar análisis de discriminación por clases y evaluar el desbalance de coberturas vegetales en la cuenca.


In [ ]:
def clasificar_cobertura(ndvi):
    if pd.isna(ndvi):
        return np.nan
    if ndvi < 0.1:
        return 'Agua/Suelo_desnudo'
    elif ndvi < 0.3:
        return 'Vegetación_escasa'
    elif ndvi < 0.6:
        return 'Vegetación_moderada'
    else:
        return 'Vegetación_densa'

df['clase_cobertura'] = df['NDVI'].apply(clasificar_cobertura)
categorias_cobertura = ['Agua/Suelo_desnudo', 'Vegetación_escasa', 'Vegetación_moderada', 'Vegetación_densa']
df['clase_cobertura'] = pd.Categorical(df['clase_cobertura'], categories=categorias_cobertura, ordered=True)
df['mes'] = df['fecha'].dt.month
df['año'] = df['fecha'].dt.year

df[['fecha', 'longitude', 'latitude', 'NDVI', 'EVI', 'clase_cobertura']].sample(5, random_state=SEED)


---
## 2. Análisis Univariado — Bandas e Índices Espectrales (Tarea 5.2)
Analizamos la distribución de probabilidad empírica de los dos índices de vegetación fundamentales:
1. **NDVI (Normalized Difference Vegetation Index):** Sensible a la clorofila activa y biomasa foliar.
2. **EVI (Enhanced Vegetation Index):** Optimizado para desacoplar el efecto de aerosoles y fondo de suelo en zonas de alta densidad foliar.


In [ ]:
# Visualización de Distribuciones: Histogramas y KDE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma NDVI
sns.histplot(df['NDVI'].dropna(), kde=True, color='#27ae60', bins=40, ax=axes[0], edgecolor='white', alpha=0.6)
axes[0].axvline(df['NDVI'].mean(), color='#c0392b', linestyle='--', linewidth=1.5, label=f"Media: {df['NDVI'].mean():.3f}")
axes[0].axvline(df['NDVI'].median(), color='#2980b9', linestyle='-', linewidth=1.5, label=f"Mediana: {df['NDVI'].median():.3f}")
axes[0].set_title("Distribución Espectral: NDVI", fontsize=12, fontweight='bold')
axes[0].set_xlabel("NDVI")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Histograma EVI
sns.histplot(df['EVI'].dropna(), kde=True, color='#2980b9', bins=40, ax=axes[1], edgecolor='white', alpha=0.6)
axes[1].axvline(df['EVI'].mean(), color='#c0392b', linestyle='--', linewidth=1.5, label=f"Media: {df['EVI'].mean():.3f}")
axes[1].axvline(df['EVI'].median(), color='#27ae60', linestyle='-', linewidth=1.5, label=f"Mediana: {df['EVI'].median():.3f}")
axes[1].set_title("Distribución Espectral: EVI", fontsize=12, fontweight='bold')
axes[1].set_xlabel("EVI")
axes[1].set_ylabel("Frecuencia")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Boxplots para Comparación y Detección de Valores Atípicos (Outliers)
plt.figure(figsize=(7, 5))
box_df = df[['NDVI', 'EVI']].dropna()
plt.boxplot([box_df['NDVI'], box_df['EVI']], tick_labels=['NDVI', 'EVI'], patch_artist=True,
            medianprops=dict(color='black', linewidth=1.5),
            boxprops=dict(facecolor='#a8dadc', color='#1d3557'),
            flierprops=dict(marker='o', markersize=3, alpha=0.4, markerfacecolor='#e63946'))
plt.title("Dispersión y Detección de Outliers: NDVI vs EVI", fontsize=12, fontweight='bold')
plt.ylabel("Magnitud del Índice")
plt.show()


In [ ]:
# Estadísticos Descriptivos Paramétricos y No Paramétricos
stats = df[['NDVI', 'EVI', 'SummaryQA']].describe().T
stats['skewness'] = [df['NDVI'].skew(), df['EVI'].skew(), df['SummaryQA'].skew()]
stats['kurtosis'] = [df['NDVI'].kurtosis(), df['EVI'].kurtosis(), df['SummaryQA'].kurtosis()]
stats.round(4)


---
## 3. Análisis Univariado — Variable Objetivo y Dinámica Temporal (Tarea 5.3)
Evaluamos la distribución de las categorías de cobertura vegetal derivadas y la serie temporal promedio multianual del Combeima.


In [ ]:
# Distribución de Clases de Cobertura
plt.figure(figsize=(8, 4.5))
counts = df['clase_cobertura'].value_counts().reindex(categorias_cobertura)
palette_clases = ['#e63946', '#f4a261', '#2a9d8f', '#1b4332']
bars = plt.bar(counts.index, counts.values, color=palette_clases, edgecolor='white', alpha=0.85)

for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df)) * 100
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 30, f"{int(yval)}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.title("Distribución de Clases de Cobertura Vegetal", fontsize=12, fontweight='bold')
plt.ylabel("Frecuencia Absoluta")
plt.ylim(0, max(counts.values) * 1.2)
plt.xticks(rotation=15, ha='right')
plt.show()

print(f"Ratio de desbalanceo (Clase Mayoritaria / Minoritaria): {counts.max() / counts.min():.2f}:1")


In [ ]:
# Serie Temporal Histórica del NDVI Promedio (2015-2024)
plt.figure(figsize=(13, 5))
ts_mean = df.groupby('fecha')['NDVI'].agg(['mean', 'std']).reset_index()
plt.plot(ts_mean['fecha'], ts_mean['mean'], color='#2d6a4f', label='NDVI Promedio Cuenca', linewidth=1.5)
plt.fill_between(ts_mean['fecha'], ts_mean['mean'] - ts_mean['std'], ts_mean['mean'] + ts_mean['std'],
                 color='#95d5b2', alpha=0.4, label='± 1 Desviación Estándar')
plt.title("Evolución Temporal del NDVI (2015 - 2024) - Cuenca Río Combeima", fontsize=12, fontweight='bold')
plt.xlabel("Fecha de Adquisición")
plt.ylabel("NDVI Promedio")
plt.legend(loc='upper right')
plt.show()


---
## 4. Análisis Bivariado — Correlaciones y Multicolinealidad (Tarea 5.4)
Identificamos el grado de colinealidad entre bandas espectrales, coordenadas geográficas, calidad satelital y temporalidad mensual.


In [ ]:
# Matriz de Correlación de Pearson
cols_corr = ['NDVI', 'EVI', 'SummaryQA', 'longitude', 'latitude', 'mes']
corr_mat = df[cols_corr].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))

plt.figure(figsize=(7, 6))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".3f", cmap="vlag", vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Matriz de Correlaciones de Pearson", fontsize=12, fontweight='bold')
plt.show()


In [ ]:
# Diagrama de Dispersión Bivariado: NDVI vs EVI
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='NDVI', y='EVI', hue='clase_cobertura', palette=palette_clases, alpha=0.6, s=25)
r_val = df['NDVI'].corr(df['EVI'])
plt.title(f"Diagrama de Dispersión NDVI vs EVI (Coeficiente r = {r_val:.3f})", fontsize=12, fontweight='bold')
plt.xlabel("NDVI")
plt.ylabel("EVI")
plt.legend(title="Clase Cobertura")
plt.show()


---
## 5. Análisis Bivariado — Features vs Clase y Ciclos Estacionales (Tarea 5.5)
Evaluamos la separabilidad de las clases de cobertura a través de los índices espectrales y el comportamiento estacional mensual.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot NDVI por Clase
sns.boxplot(data=df, x='clase_cobertura', y='NDVI', hue='clase_cobertura', palette=palette_clases, legend=False, ax=axes[0],
            boxprops=dict(alpha=0.8), flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0].set_title("NDVI según Clase de Cobertura", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Clase")
axes[0].tick_params(axis='x', rotation=15)

# Boxplot EVI por Clase
sns.boxplot(data=df, x='clase_cobertura', y='EVI', hue='clase_cobertura', palette=palette_clases, legend=False, ax=axes[1],
            boxprops=dict(alpha=0.8), flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_title("EVI según Clase de Cobertura", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Clase")
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
# Ciclo Estacional Mensual (Violinplot)
plt.figure(figsize=(11, 5))
sns.violinplot(data=df, x='mes', y='NDVI', hue='mes', palette="YlGn", legend=False, inner="quartile", linewidth=1.2)
plt.title("Ciclo Estacional Anual: Distribución Mensual del NDVI", fontsize=12, fontweight='bold')
plt.xlabel("Mes del Año")
plt.ylabel("NDVI")
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
plt.xticks(ticks=range(12), labels=meses)
plt.show()


---
## 6. Análisis Multivariado — Reducción de Dimensionalidad con PCA (Tarea 5.6)
Aplicamos Análisis de Componentes Principales (PCA) sobre las variables estandarizadas para entender la compresión espectral y varianza explicada.


In [ ]:
cols_pca = ['NDVI', 'EVI', 'SummaryQA', 'longitude', 'latitude']
df_pca = df[cols_pca].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)

pca = PCA(n_components=len(cols_pca), random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_
cum_var = np.cumsum(var_exp)

# Scree Plot de Varianza Explicada
plt.figure(figsize=(8, 4.5))
plt.bar(range(1, len(var_exp)+1), var_exp * 100, color='#457b9d', alpha=0.7, label='Varianza Individual')
plt.plot(range(1, len(var_exp)+1), cum_var * 100, marker='o', color='#e63946', linewidth=2, label='Varianza Acumulada')
plt.axhline(85, color='gray', linestyle='--', label='Criterio 85% Varianza')

for i, (ve, cv) in enumerate(zip(var_exp, cum_var)):
    plt.text(i+1, (ve*100)/2, f"{ve*100:.1f}%", ha='center', va='center', color='white', fontweight='bold', fontsize=9)

plt.title("PCA: Varianza Explicada por Componente Principal", fontsize=12, fontweight='bold')
plt.xlabel("Componente Principal")
plt.ylabel("% Varianza Explicada")
plt.xticks(range(1, len(var_exp)+1), [f"PC{i}" for i in range(1, len(var_exp)+1)])
plt.ylim(0, 105)
plt.legend(loc='center right')
plt.show()

print(f"Número de componentes requeridos para superar el 85% de varianza: {np.argmax(cum_var >= 0.85) + 1}")


In [ ]:
# Proyección 2D del Espacio Latente
df_pca_plot = pd.DataFrame(X_pca[:, :2], columns=['PC1', 'PC2'])
df_pca_plot['clase'] = df.loc[df_pca.index, 'clase_cobertura'].values

plt.figure(figsize=(8.5, 5.5))
sns.scatterplot(data=df_pca_plot, x='PC1', y='PC2', hue='clase', palette=palette_clases, alpha=0.6, s=25)
plt.title(f"Proyección PCA 2D (PC1: {var_exp[0]*100:.1f}% + PC2: {var_exp[1]*100:.1f}%)", fontsize=12, fontweight='bold')
plt.xlabel(f"PC1 ({var_exp[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({var_exp[1]*100:.1f}%)")
plt.legend(title="Clase Cobertura")
plt.show()


---
## 7. Análisis Geoespacial — Distribución Espacial de la Cuenca Combeima (Tarea 5.7)
Visualizamos la distribución geográfica de los píxeles muestreados en la cuenca y su NDVI promedio multianual. Adicionalmente, verificamos la inicialización con Google Earth Engine para mapas satelitales interactivos.


In [ ]:
# Mapa Espacial Estático con Matplotlib (Respaldo Robusto)
pixeles = df.groupby(['longitude', 'latitude']).agg({
    'NDVI': 'mean',
    'EVI': 'mean',
    'fecha': 'count'
}).reset_index().rename(columns={'fecha': 'n_muestras'})

plt.figure(figsize=(8.5, 6.5))
scatter = plt.scatter(pixeles['longitude'], pixeles['latitude'],
                      c=pixeles['NDVI'], cmap='RdYlGn', s=120, edgecolors='black', linewidth=0.8, alpha=0.9)
cbar = plt.colorbar(scatter)
cbar.set_label('NDVI Promedio Multianual (2015-2024)', fontsize=10)
plt.title("Distribución Espacial de Muestras Satelitales — Cuenca Combeima", fontsize=12, fontweight='bold')
plt.xlabel("Longitud (WGS84)")
plt.ylabel("Latitud (WGS84)")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


In [ ]:
# Diagnóstico de Conexión a Google Earth Engine
try:
    import ee
    ee.Initialize(project='skyfussionanalytics')
    print("[OK] Conexión exitosa a Google Earth Engine con el proyecto 'skyfussionanalytics'")
except Exception as e:
    print(f"ℹ GEE no inicializado en este paso interactivo: {e}")
    print("Se conserva la visualización geoespacial cartesiana de alta precisión como entregable.")


---
## 8. Informe de Hallazgos y Recomendaciones para Modelado (Tarea 5.8)
Se presenta la consolidación de los 5 hallazgos analíticos con su impacto y recomendación técnica para el entrenamiento de la Red Neuronal Recurrente.


In [ ]:
# Carga y presentación del informe estructurado de hallazgos
df_hallazgos_view = pd.read_csv(REPORTS_DIR / 'hallazgos_eda.csv')
pd.set_option('display.max_colwidth', None)
df_hallazgos_view[['id', 'dimension', 'hallazgo', 'recomendacion', 'prioridad']]


---
## 9. Exportación y Metadatos para la Fase de Modelado (Tarea 5.9)
Exportamos las especificaciones de características recomendadas, hiperparámetros del preprocesamiento y criterios de estratificación espacial en formato JSON para consumo automático por el pipeline de entrenamiento (`rnn_combeima/rnn/train.py`).


In [ ]:
with open(REPORTS_DIR / 'features_modelado.json', 'r', encoding='utf-8') as f:
    config_features = json.load(f)

print("=== ESPECIFICACIÓN TÉCNICA PARA MODELADO RNA ===")
print(json.dumps(config_features, indent=2, ensure_ascii=False))
